# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Lasso
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import time

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [ ]:
bank_ds = pd.read_csv('/content/bank-additional-full.csv', sep = ';')
#bank_ds = bank_ds.sample(frac = 0.20, random_state = 42)
#bank_ds = pd.read_csv('/content/bank-additional.csv', sep = ';')

In [ ]:
#look at the sample of bank_smpl/bank_full DataFrame and see .info()
print(bank_ds.head())
bank_ds.info()

   age        job  marital    education  default housing loan    contact  \
0   56  housemaid  married     basic.4y       no      no   no  telephone   
1   57   services  married  high.school  unknown      no   no  telephone   
2   37   services  married  high.school       no     yes   no  telephone   
3   40     admin.  married     basic.6y       no      no   no  telephone   
4   56   services  married  high.school       no      no  yes  telephone   

  month day_of_week  ...  campaign  pdays  previous     poutcome emp.var.rate  \
0   may         mon  ...         1    999         0  nonexistent          1.1   
1   may         mon  ...         1    999         0  nonexistent          1.1   
2   may         mon  ...         1    999         0  nonexistent          1.1   
3   may         mon  ...         1    999         0  nonexistent          1.1   
4   may         mon  ...         1    999         0  nonexistent          1.1   

   cons.price.idx  cons.conf.idx  euribor3m  nr.employed

**DN comments:**

I used both the full dataset (100%) and the smaller dataset (10%) in this assignment. In the Model Comparison section of this assignment, I experienced a number of time-outs during steps that had multiple models running in a pipeline.

My analysis proved that it timed-out during the SVC model processing. As a result I used a smaller dataset and also altered hyperparameters to get past that issue.

I was eventually able to use the larger (100%) dataset and have documented the results towards the bottom of this workbook.

### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [ ]:
#create X and y datasets
#drop duration column
X = bank_ds.drop(columns = ["y", "duration"])
y = bank_ds["y"]

#how balanced are the classes?
class_counts = pd.DataFrame(y.value_counts(normalize = True))
print(class_counts)

#plot the balance
#bar_plot = px.bar(class_counts, y = "proportion", x = "y")
bar_plot = px.bar(class_counts, y = "proportion")

bar_plot.update_xaxes(
    title_text = "Data Classes"
)
bar_plot.update_yaxes(
    title_text = "Proportion %",
    showgrid = True
)
bar_plot.update_layout(
    title_text = "Data Class Balance"
)
bar_plot.show()



     proportion
y              
no     0.887346
yes    0.112654


**DN comments:**

As can be seen in the cell above, the classes heavily weighted to "no" and are therefore unbalanced.

Also, I dropped the duration column from the dataset per the recommendation in the dataset documentation.

### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

**DN comments:**

European banks at the time of this marketing campaign were operating in somewhat of a financial crisis and were facing pressure to increase financial assets. As a solution to this business problem an adopted strategy was to offer long-term
deposit applications with good interest rates. These term deposits were fixed-interest savings accounts where you leave a lump sum of money untouched for a set period, ranging from a few months to several years. In exchange for locking your funds away, the bank pays you a guaranteed, often higher interest rate than a standard savings account.

The business goal of the marketing campaign was to find a model that can explain the success of a contact; whether the client subscribes the deposit. Such a model could increase campaign efficiency by identifying the main characteristics that affect success, helping in a better management of the available resources such as human effort, phone calls, time spent, and selection of a high-quality and affordable set of potential buying customers.

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [ ]:
# Encode the target variable
le = LabelEncoder()
y = le.fit_transform(y)

#determine data type and separate object from float and integer
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()


### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [ ]:
#create train and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)

### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [ ]:

#use DummyClassifier() to get Baseline scoring
dummy_clf = DummyClassifier().fit(X_train, y_train)

baseline_train_score = dummy_clf.score(X_train, y_train)
baseline_test_score = dummy_clf.score(X_test, y_test)
print(f"Baseline train score: {baseline_train_score}")
print(f"Baseline test score: {baseline_test_score}")


Baseline train score: 0.8872394297804447
Baseline test score: 0.8875940762320952


In [ ]:
print(X_test.head())
print(y_test)


       age          job  marital    education  default housing loan  \
32884   57   technician  married  high.school       no      no  yes   
3169    55      unknown  married      unknown  unknown     yes   no   
32206   33  blue-collar  married     basic.9y       no      no   no   
9403    36       admin.  married  high.school       no      no   no   
14020   27    housemaid  married  high.school       no     yes   no   

         contact month day_of_week  campaign  pdays  previous     poutcome  \
32884   cellular   may         mon         1    999         1      failure   
3169   telephone   may         thu         2    999         0  nonexistent   
32206   cellular   may         fri         1    999         1      failure   
9403   telephone   jun         fri         4    999         0  nonexistent   
14020   cellular   jul         fri         2    999         0  nonexistent   

       emp.var.rate  cons.price.idx  cons.conf.idx  euribor3m  nr.employed  
32884          -1.8        

### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [ ]:
#this section does Logistic Regression only
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

models = {
    'logisticregression': (LogisticRegression(max_iter=1000), {'logisticregression__C': [0.1, 1, 10]})
    }

results = []

for name, (model, params) in models.items():
    # Create a pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        (name, model)
    ])

    # Perform grid search
    grid_search = GridSearchCV(pipeline, param_grid=params, cv=5, n_jobs=-1)

    # Fit the model and time it
    start_time = time.time()
    grid_search.fit(X_train, y_train)
    fit_time = (time.time() - start_time) / len(grid_search.cv_results_['mean_fit_time'])

    # Get the best estimator
    best_model = grid_search.best_estimator_

    #get the the best_params_
    best_params = grid_search.best_params_

    # Evaluate on training and test sets
    train_score = best_model.score(X_train, y_train)
    test_score = best_model.score(X_test, y_test)

    # Append the results
    results.append([name, train_score, test_score, fit_time, best_params])

# Create the results DataFrame
results_df = pd.DataFrame(results, columns=["model", "train score", "test score", "average fit time", "best params"])
results_df.set_index("model", inplace=True)


#pred = grid_search.predict(X_test)
#prob = grid_search.predict_proba(X_test)

#print(classification_report(y_test, pred))

### Problem 9: Score the Model

What is the accuracy of your model?

In [ ]:
print(results_df)

                    train score  test score  average fit time  \
model                                                           
logisticregression     0.900489    0.901109          3.237634   

                                       best params  
model                                               
logisticregression  {'logisticregression__C': 0.1}  


### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [ ]:

#this section does comparison of KNN, Logistic Regression,SVM and Decision Trees
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

models = {
    'knn': (KNeighborsClassifier(), {'knn__n_neighbors': [5]}),
    'logisticregression': (LogisticRegression(max_iter=1000), {'logisticregression__C': [0.1, 1, 10]}),
    #'svc': (SVC(), {'svc__C': [0.1, 1, 10], 'svc__coef0': [1], 'svc__degree': [8], 'svc__kernel': ['linear', 'rbf', 'poly']}),
    'svc': (SVC(), {'svc__C': [0.1], 'svc__kernel': ['linear', 'rbf']}),
    #'decisiontreeclassifier': (DecisionTreeClassifier(), {'decisiontreeclassifier__max_depth': [5, 10, 15]})
    'decisiontreeclassifier': (DecisionTreeClassifier(), {'decisiontreeclassifier__max_depth': [5]})

}

results = []

for name, (model, params) in models.items():
    # Create a pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        (name, model)
    ])

    # Perform grid search
    grid_search = GridSearchCV(pipeline, param_grid=params, cv=5, n_jobs=-1)

    # Fit the model and time it
    start_time = time.time()
    grid_search.fit(X_train, y_train)
    fit_time = (time.time() - start_time) / len(grid_search.cv_results_['mean_fit_time'])

    # Get the best estimator
    best_model = grid_search.best_estimator_

    #get the the best_params_
    best_params = grid_search.best_params_
    print(best_params)

    # Evaluate on training and test sets
    train_score = best_model.score(X_train, y_train)
    test_score = best_model.score(X_test, y_test)

    # Append the results
    results.append([name, train_score, test_score, fit_time, best_params])

# Create the results DataFrame
results_df = pd.DataFrame(results, columns=["model", "train score", "test score", "average fit time", "best params"])
results_df.set_index("model", inplace=True)

print(results_df)


{'knn__n_neighbors': 5}
{'logisticregression__C': 0.1}
{'svc__C': 0.1, 'svc__kernel': 'rbf'}
{'decisiontreeclassifier__max_depth': 5}
                        train score  test score  average fit time  \
model                                                               
knn                        0.913080    0.889698          5.927349   
logisticregression         0.900489    0.901109          2.387560   
svc                        0.899587    0.899247        179.455645   
decisiontreeclassifier     0.904131    0.900218          1.292465   

                                                     best params  
model                                                             
knn                                      {'knn__n_neighbors': 5}  
logisticregression                {'logisticregression__C': 0.1}  
svc                        {'svc__C': 0.1, 'svc__kernel': 'rbf'}  
decisiontreeclassifier  {'decisiontreeclassifier__max_depth': 5}  


### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

In [ ]:
#this section does comparison of KNN, Logistic Regression,SVM and Decision Trees with adjusted hyperparameters
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

models = {
    'knn': (KNeighborsClassifier(), {'knn__n_neighbors': [3, 5, 7]}),
    'logisticregression': (LogisticRegression(max_iter=1000), {'logisticregression__C': [0.1, 1, 10]}),
    #'svc': (SVC(), {'svc__C': [0.1, 1, 10], 'svc__coef0': [1], 'svc__degree': [8], 'svc__kernel': ['linear', 'rbf', 'poly']}),
    'svc': (SVC(), {'svc__C': [0.1, 1], 'svc__kernel': ['linear', 'rbf']}),
    #'decisiontreeclassifier': (DecisionTreeClassifier(), {'decisiontreeclassifier__max_depth': [5, 10, 15]})
    'decisiontreeclassifier': (DecisionTreeClassifier(), {'decisiontreeclassifier__max_depth': [5, 10]})

}

results = []

for name, (model, params) in models.items():
    # Create a pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        (name, model)
    ])

    # Perform grid search
    grid_search = GridSearchCV(pipeline, param_grid=params, cv=5, n_jobs=-1)

    # Fit the model and time it
    start_time = time.time()
    grid_search.fit(X_train, y_train)
    fit_time = (time.time() - start_time) / len(grid_search.cv_results_['mean_fit_time'])

    # Get the best estimator
    best_model = grid_search.best_estimator_

    #get the the best_params_
    best_params = grid_search.best_params_
    print(best_params)

    # Evaluate on training and test sets
    train_score = best_model.score(X_train, y_train)
    test_score = best_model.score(X_test, y_test)

    # Append the results
    results.append([name, train_score, test_score, fit_time, best_params])

# Create the results DataFrame
results_df = pd.DataFrame(results, columns=["model", "train score", "test score", "average fit time", "best params"])
results_df.set_index("model", inplace=True)

print(results_df)

{'knn__n_neighbors': 7}
{'logisticregression__C': 0.1}


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning:

A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.



{'svc__C': 1, 'svc__kernel': 'rbf'}
{'decisiontreeclassifier__max_depth': 5}
                        train score  test score  average fit time  \
model                                                               
knn                        0.909889    0.892450          6.454683   
logisticregression         0.900489    0.901109          2.822664   
svc                        0.904825    0.900704        437.197558   
decisiontreeclassifier     0.904131    0.900218          1.228947   

                                                     best params  
model                                                             
knn                                      {'knn__n_neighbors': 7}  
logisticregression                {'logisticregression__C': 0.1}  
svc                          {'svc__C': 1, 'svc__kernel': 'rbf'}  
decisiontreeclassifier  {'decisiontreeclassifier__max_depth': 5}  


In [ ]:
#this section will read and prep accumulated results
results_df = pd.read_csv('/content/Assign 17-1 Results.csv', sep = ',')
results_df.fillna("", inplace=True)
#results_df.set_index("model", inplace=True)

#print(results_df.info())
#print(results_df)

/tmp/ipykernel_630/3371354326.py:3: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.



In [ ]:
#plot the accumulated results data in a table
table_plot = go.Figure(data = [go.Table(
            header = dict(values = list(results_df.columns),
                  align = "left"),
            cells = dict(values = [results_df["model"],
                           results_df["train score"],
                           results_df["test score"],
                           results_df["average fit time"],
                           results_df["best params"],
                           results_df["Test Notes"]],
                  align = "left"))
])

table_plot.update_layout(
    title_text = "Assignment 17.1 Results",
    title_x = 0.5)

table_plot.show()

**DN comments:**

In the table above I am displaying results that I accumulated over multilpe iterations. The table is scrollable when you place the cursor/pointer on the table. This will allow you to display all results.


**DN comments:**

All test scores exceeded the DummyClassifier Baseline. Further I looked for cases that had high test scores where the Test Score exceeded the Train Score and found this trend: **Logistic Regression** consistently had Test Scores that increased over Train Scores and **SVC** intermittently had Test Scores that exceeded Train Scores, or were very close.

With respect to Average Fit Time, decisiontreeclassifier consistently had the smallest amount of time. The optimum max_depth 5 was consistently selected. The highest Average Fit Time was SVC. In one test using the 100% dataset the multiple comparison between lowest and highest Average Fit Time was ~378x.

I timed-out on 4 iterations when processing the SVC model when using the 100% dataset. When I ran with the 10% dataset I ran to conclusion and the C=1 hyperparameter (regularization) was chosen in best_params. I then re-ran SVC using the 100% dataset and ran to conclusion but I kept values for C less than or equal to 1.0. The learning was to be aware of the compute required to execute the models and also highlighted the potentially significant difference in compute across the models.

Based on my testing of the models I would recommend Logistic Regression as the best performer with respect to performance and compute efficiency. Additional testing should be done with SVC using variations of hyperparameters. The main consideration with SVC is the amount of compute required relative to the other models.

##### Questions